# Glass Type Classification

This notebook explores the UCI Glass Identification dataset and builds a classification model
to predict the type of glass based on its chemical composition.

**Dataset Attributes:**
- **RI** – Refractive Index
- **Na** – Sodium
- **Mg** – Magnesium
- **Al** – Aluminum
- **Si** – Silicon
- **K** – Potassium
- **Ca** – Calcium
- **Ba** – Barium
- **Fe** – Iron
- **Type** – Glass type (target variable)

**Glass Types:**
1. Building Windows – Float Processed
2. Building Windows – Non-Float Processed
3. Vehicle Windows – Float Processed
4. Vehicle Windows – Non-Float Processed *(not present in this dataset)*
5. Containers
6. Tableware
7. Headlamps

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    ConfusionMatrixDisplay
)
from sklearn.preprocessing import StandardScaler

# Interactive widget support
import ipywidgets as widgets
from IPython.display import display, HTML

# Consistent plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)

print('All libraries loaded successfully!')

## 2. Data Loading

Load the glass dataset from the `data/` directory.

In [ ]:
# Load the dataset
df = pd.read_csv('../data/glass.csv')

# Map numeric type codes to descriptive labels
GLASS_TYPE_LABELS = {
    1: 'building_windows_float_processed',
    2: 'building_windows_non_float_processed',
    3: 'vehicle_windows_float_processed',
    4: 'vehicle_windows_non_float_processed',
    5: 'containers',
    6: 'tableware',
    7: 'headlamps',
}

df['Type_Label'] = df['Type'].map(GLASS_TYPE_LABELS)

print(f'Dataset shape: {df.shape}')
df.head(10)

## 3. Exploratory Data Analysis (EDA)

### 3.1 Dataset Overview

In [ ]:
print('=== Dataset Info ===')
df.info()
print()
print('=== Missing Values ===')
print(df.isnull().sum())

In [ ]:
print('=== Descriptive Statistics ===')
df.describe().round(4)

### 3.2 Class Distribution

In [ ]:
class_counts = df['Type_Label'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
class_counts.plot(kind='bar', ax=axes[0], color=sns.color_palette('muted', len(class_counts)))
axes[0].set_title('Glass Type Distribution (Count)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Glass Type')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Pie chart
axes[1].pie(
    class_counts.values,
    labels=class_counts.index,
    autopct='%1.1f%%',
    colors=sns.color_palette('muted', len(class_counts)),
    startangle=140
)
axes[1].set_title('Glass Type Distribution (%)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print(class_counts.to_frame('Count'))

### 3.3 Feature Distributions

In [ ]:
features = ['RI', 'Na', 'Mg', 'Al', 'Si', 'K', 'Ca', 'Ba', 'Fe']

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, feat in enumerate(features):
    axes[i].hist(df[feat], bins=25, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(f'Distribution of {feat}', fontweight='bold')
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('Frequency')

plt.suptitle('Feature Distributions', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 3.4 Feature Distributions by Glass Type

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 13))
axes = axes.flatten()

palette = sns.color_palette('muted', df['Type_Label'].nunique())

for i, feat in enumerate(features):
    for j, (gtype, grp) in enumerate(df.groupby('Type_Label')):
        axes[i].hist(grp[feat], bins=20, alpha=0.55, label=gtype, color=palette[j])
    axes[i].set_title(f'{feat} by Glass Type', fontweight='bold')
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('Frequency')

# Single shared legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.02), fontsize=9)
plt.suptitle('Feature Distributions by Glass Type', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.5 Correlation Heatmap

In [ ]:
corr = df[features + ['Type']].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    linewidths=0.5,
    square=True
)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.6 Box Plots – Feature vs Glass Type

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 13))
axes = axes.flatten()

for i, feat in enumerate(features):
    df.boxplot(
        column=feat,
        by='Type_Label',
        ax=axes[i],
        vert=True,
        patch_artist=True
    )
    axes[i].set_title(f'{feat} by Glass Type', fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=45)

plt.suptitle('Box Plots: Feature Values by Glass Type', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Data Preprocessing

The dataset requires no encoding since all features are already numerical.
We split into features (`X`) and target (`y`), then create training and test sets.

In [ ]:
# Separate features and target
X = df[features]
y = df['Type']

# Train/test split (80/20, stratified to preserve class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature scaling (important for distance-based models like SVC and KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Training set size : {X_train.shape[0]} samples')
print(f'Test set size     : {X_test.shape[0]} samples')
print(f'Number of features: {X_train.shape[1]}')
print(f'Number of classes : {y.nunique()}')

## 5. Model Training

We evaluate four common classifiers and select the best performer based on accuracy.

In [ ]:
models = {
    'Random Forest'           : RandomForestClassifier(n_estimators=200, random_state=42),
    'Gradient Boosting'       : GradientBoostingClassifier(n_estimators=200, random_state=42),
    'Support Vector Machine'  : SVC(kernel='rbf', C=10, gamma='scale', random_state=42),
    'K-Nearest Neighbours'    : KNeighborsClassifier(n_neighbors=5),
}

results = {}

print(f'{'Model':<30} {'CV Accuracy':>12} {'Test Accuracy':>14}')
print('-' * 58)

for name, model in models.items():
    # Use scaled data for SVC and KNN, raw for tree-based models
    use_scaled = name in ['Support Vector Machine', 'K-Nearest Neighbours']
    Xtr = X_train_scaled if use_scaled else X_train
    Xte = X_test_scaled  if use_scaled else X_test

    # 5-fold cross-validation on training set
    cv_scores = cross_val_score(model, Xtr, y_train, cv=5, scoring='accuracy')

    # Train and evaluate on test set
    model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)
    test_acc = accuracy_score(y_test, y_pred)

    results[name] = {
        'model'     : model,
        'cv_mean'   : cv_scores.mean(),
        'cv_std'    : cv_scores.std(),
        'test_acc'  : test_acc,
        'y_pred'    : y_pred,
        'use_scaled': use_scaled,
    }

    print(f'{name:<30} {cv_scores.mean():>11.4f}  {test_acc:>13.4f}')

print('-' * 58)

### 5.1 Model Accuracy Comparison

In [ ]:
model_names = list(results.keys())
cv_accs     = [results[m]['cv_mean'] for m in model_names]
test_accs   = [results[m]['test_acc'] for m in model_names]

x = np.arange(len(model_names))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 6))
bars1 = ax.bar(x - width/2, cv_accs,   width, label='CV Accuracy (train)',  color='steelblue')
bars2 = ax.bar(x + width/2, test_accs, width, label='Test Accuracy',        color='darkorange')

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=15, ha='right')
ax.set_ylim(0, 1.05)
ax.legend()
ax.bar_label(bars1, fmt='%.3f', padding=3)
ax.bar_label(bars2, fmt='%.3f', padding=3)

plt.tight_layout()
plt.show()

## 6. Model Evaluation

We select the best model by test accuracy and examine detailed performance metrics.

In [ ]:
# Select the best model by test accuracy
best_name = max(results, key=lambda m: results[m]['test_acc'])
best = results[best_name]

print(f'Best model: {best_name}')
print(f'Test Accuracy: {best["test_acc"]:.4f}  ({best["test_acc"]*100:.2f}%)')

### 6.1 Classification Report

In [ ]:
y_pred_best = best['y_pred']
target_names = [GLASS_TYPE_LABELS[t] for t in sorted(y_test.unique())]

print(f'Classification Report – {best_name}\n')
print(classification_report(y_test, y_pred_best, target_names=target_names))

### 6.2 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred_best, labels=sorted(y_test.unique()))

fig, ax = plt.subplots(figsize=(9, 7))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=target_names
)
disp.plot(ax=ax, cmap='Blues', colorbar=True)
ax.set_title(f'Confusion Matrix – {best_name}', fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 6.3 Feature Importance (Random Forest)

In [ ]:
rf_model = results['Random Forest']['model']
importances = pd.Series(rf_model.feature_importances_, index=features).sort_values(ascending=False)

plt.figure(figsize=(9, 5))
importances.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Feature Importance – Random Forest', fontsize=13, fontweight='bold')
plt.ylabel('Importance Score')
plt.xlabel('Feature')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print(importances.to_frame('Importance').round(4))

## 7. Predicting Glass Type

Convert numeric prediction codes to descriptive glass type labels.

In [ ]:
def predict_glass_type(model, scaler_obj, sample: dict, use_scaled: bool = False) -> str:
    """
    Predict the glass type label for a single sample.

    Parameters
    ----------
    model       : trained sklearn classifier
    scaler_obj  : fitted StandardScaler (used when use_scaled=True)
    sample      : dict of {feature_name: value}
    use_scaled  : whether to apply the scaler before prediction

    Returns
    -------
    str : descriptive glass type label
    """
    X_sample = pd.DataFrame([sample])[features]
    if use_scaled:
        X_sample = scaler_obj.transform(X_sample)
    code = model.predict(X_sample)[0]
    return GLASS_TYPE_LABELS.get(code, f'Unknown (code {code})')


# --- Example prediction ---
sample = {
    'RI': 1.51766, 'Na': 13.21, 'Mg': 3.69, 'Al': 1.29,
    'Si': 72.61,   'K' : 0.57,  'Ca': 8.22, 'Ba': 0.00, 'Fe': 0.00
}

predicted_label = predict_glass_type(
    best['model'], scaler, sample, best['use_scaled']
)
print(f'Sample input     : {sample}')
print(f'Predicted type   : {predicted_label}')

## 8. Interactive Prediction Widget

Use the sliders below to enter chemical composition values and get an instant glass type prediction.

In [ ]:
# ── Widget definition ────────────────────────────────────────────────────────
feature_ranges = {
    'RI': (1.510, 1.535, 0.0001, 1.518),
    'Na': (10.0,  17.5,  0.01,   13.5),
    'Mg': (0.0,   4.5,   0.01,   3.5),
    'Al': (0.5,   3.5,   0.01,   1.3),
    'Si': (69.0,  75.5,  0.01,   72.8),
    'K' : (0.0,   6.5,   0.01,   0.6),
    'Ca': (5.0,   16.5,  0.01,   8.9),
    'Ba': (0.0,   3.5,   0.01,   0.0),
    'Fe': (0.0,   0.55,  0.001,  0.0),
}

sliders = {
    feat: widgets.FloatSlider(
        value=default,
        min=lo, max=hi, step=step,
        description=feat + ':',
        style={'description_width': '40px'},
        layout=widgets.Layout(width='450px'),
        readout_format='.4f' if feat == 'RI' else '.3f'
    )
    for feat, (lo, hi, step, default) in feature_ranges.items()
}

output_label = widgets.HTML(value='<b>Predicted glass type will appear here.</b>')

predict_button = widgets.Button(
    description='Predict Glass Type',
    button_style='primary',
    layout=widgets.Layout(width='200px', height='36px')
)

def on_predict_clicked(_):
    sample = {feat: sliders[feat].value for feat in features}
    label = predict_glass_type(best['model'], scaler, sample, best['use_scaled'])
    output_label.value = (
        f'<div style="padding:10px; background:#e8f4f8; border-radius:6px;'
        f' border-left:4px solid #2196F3; font-size:15px;">'
        f'<b>Predicted Glass Type:</b><br/>'
        f'<span style="color:#1565C0; font-size:16px;">'
        f'{label.replace("_", " ").title()}</span></div>'
    )

predict_button.on_click(on_predict_clicked)

# Layout
slider_box   = widgets.VBox(list(sliders.values()))
widget_panel = widgets.VBox(
    [widgets.HTML('<h4>Enter Chemical Composition</h4>'),
     slider_box, predict_button, output_label],
    layout=widgets.Layout(padding='15px', border='1px solid #ccc', border_radius='8px')
)
display(widget_panel)

## 9. Summary

| Step | Details |
|------|---------|
| Dataset | UCI Glass Identification – 214 samples, 9 features, 6 active classes |
| No encoding required | All features are already numerical |
| Models compared | Random Forest, Gradient Boosting, SVM, KNN |
| Best model | Selected automatically by test accuracy |
| Metrics | Accuracy, precision, recall, F1-score, confusion matrix |
| Interactive widget | Real-time glass type prediction via ipywidgets |

The **Random Forest** and **Gradient Boosting** classifiers typically achieve the highest
accuracy on this dataset. Magnesium (Mg) and Barium (Ba) tend to be the most
discriminative features for separating glass types.